In [ ]:
import os
import shutil

# Remove existing directory if it exists
ROOT = '/kaggle/working/mini-gpt'
if os.path.exists(ROOT):
    shutil.rmtree(ROOT)

# Clone the repository from GitHub
!git clone https://github.com/maariogutierrez/mini-gpt.git {ROOT}
os.chdir(ROOT)

REQUIREMENTS = 'https://raw.githubusercontent.com/maariogutierrez/mini-gpt/main/requirements.txt'
!pip install -r $REQUIREMENTS

print("Logging into Weights & Biases (wandb). Follow the prompts.")
import wandb
wandb.login(key='')

# Store outputs in Google Drive, but keep code in cloned repo
OUTPUT_ROOT = '/kaggle/working/output'
os.makedirs(OUTPUT_ROOT, exist_ok=True)

DATA_DIR = os.path.join(OUTPUT_ROOT, 'data')
EXPORTS_DIR = os.path.join(OUTPUT_ROOT, 'exports')
LOGS_DIR = os.path.join(OUTPUT_ROOT, 'logs')
CHECKPOINTS_DIR = os.path.join(OUTPUT_ROOT, 'checkpoints')

for d in [DATA_DIR, EXPORTS_DIR, LOGS_DIR, CHECKPOINTS_DIR]:
    os.makedirs(d, exist_ok=True)

print("\n--- Kaggle Setup Summary ---")
print(f"Code repository: {ROOT}")
print(f"Output directory: {OUTPUT_ROOT}")
print(f"Current Working Directory: {os.getcwd()}")

In [ ]:
import gdown
import os

# Replace these IDs with your actual File IDs from Google Drive
files_to_download = {
    'train.bin': '1wRP5jrk6csXbvFLDJz1YtEAZLKZbQc0r',
    'val.bin': '1QZNte5AFwJZ9MrmcOUu8LCKQR6e40MMI',
    'checkpoint.pt': '1hhXM3XSkhWbwVUV-9NRmq4JAFLvs29tu'
}

# Download the training data
print("Downloading train.bin...")
gdown.download(f'https://drive.google.com/uc?id={files_to_download["train.bin"]}', 
               '/kaggle/working/output/data/train.bin', quiet=False)

print("Downloading val.bin...")
gdown.download(f'https://drive.google.com/uc?id={files_to_download["val.bin"]}', 
               '/kaggle/working/output/data/val.bin', quiet=False)

# Download the checkpoint
print("Downloading checkpoint...")
gdown.download(f'https://drive.google.com/uc?id={files_to_download["checkpoint.pt"]}', 
               '/kaggle/working/output/checkpoints/checkpoint.pt', quiet=False)

print("Files successfully moved to Kaggle!")

In [ ]:

from model.architecture.gpt import GPT, GPTConfig
from model.training.trainer import Trainer
from model.training.dataset import TokenDataset
import torch


# Model configuration - balanced for efficient training
model_config = GPTConfig(
    vocab_size=50304,      
    block_size=256,        # Context window
    n_layer=12,            # 12 transformer layers
    n_head=12,             # 12 attention heads
    n_embd=768,            # 768 embedding dimension
    dropout=0.1
)

print("Creating model...")
model = GPT(model_config)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Device: {device}")

local_train_bin = os.path.join(DATA_DIR, 'train.bin')
local_val_bin = os.path.join(DATA_DIR, 'val.bin')

print("Loading datasets...")
train_dataset = TokenDataset(local_train_bin, model_config.block_size)
val_dataset = TokenDataset(local_val_bin, model_config.block_size)
print(f"Train dataset: {len(train_dataset)} samples")
print(f"Val dataset: {len(val_dataset)} samples")

trainer = Trainer(
    model=model,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    batch_size=16,
    accum_steps=32,        # 16 × 32 = 512 effective batch size
    learning_rate=6e-4,    # Moderate learning rate
    weight_decay=0.1,
    warmup_steps=1200,      # 1200 warmup steps
    max_steps=60000,        # Train for 60000 steps
    grad_clip=1.0,
    device=device,
    checkpoint_dir=CHECKPOINTS_DIR,
    wandb_project="mini-gpt",
    use_mixed_precision=True,
)

print(f"\nTraining configuration:")
print(f"  Effective batch size: {trainer.effective_batch_size}")
print(f"  Max steps: {trainer.max_steps}")

In [ ]:

# Run training
print("=" * 60)
print("Starting training session")
print("=" * 60)

checkpoint_path = os.path.join(CHECKPOINTS_DIR, "checkpoint.pt")
trainer.load_checkpoint(checkpoint_path)
trainer.train()

print("\n" + "=" * 60)
print("Training session complete!")
print("=" * 60)
print(f"Final step: {trainer.global_step}")
print(f"Best validation loss: {trainer.best_val_loss:.4f}")


In [ ]:
import os
import torch

from model.architecture.gpt import GPT, GPTConfig
from model.architecture.tokenizer import CustomTokenizer

tokenizer = CustomTokenizer()

model_config = GPTConfig(
    vocab_size=50304,      
    block_size=512,         # Context window
    n_layer=12,             # 8 transformer layers
    n_head=12,              # 8 attention heads
    n_embd=768,             # 512 embedding dimension
    dropout=0.1
)

device = "cuda" if torch.cuda.is_available() else "cpu"

# Recreate the model architecture
model = GPT(model_config).to(device)

# Load best checkpoint weights
best_checkpoint_path = os.path.join(CHECKPOINTS_DIR, "best_model.pt")
checkpoint = torch.load(best_checkpoint_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print(f"Loaded best model from: {best_checkpoint_path}")
print(f"Best validation loss: {checkpoint['best_val_loss']:.4f}")

# Generate text
prompt = "Once upon a time"
idx = torch.tensor([tokenizer.encode(prompt)], device=device)

generated = model.generate(idx, max_new_tokens=100, temperature=0.8)
text = tokenizer.decode(generated[0].tolist())

print(text)